# Python Basics — For Engineers Who Already Know How to Code

## Who This Notebook Is For

This notebook is **not** "what is a variable."

It is:
- Python's quirks vs other languages (the stuff that bites experienced engineers)
- The data structures ML code uses constantly and *why*
- Pythonic patterns — list comprehensions, unpacking, generators
- The things We'll see in every sklearn/numpy/pandas codebase

**Reading time:** ~45 minutes
**Goal:** Write Python that looks like it was written by a Python developer, not translated from Java

---

## What We Cover

| Section | Topic | Why It Matters for ML |
|---|---|---|
| 1 | Types, Variables & the Python Data Model | `None` vs `NaN` will break our pipeline |
| 2 | Lists, Tuples & the Sequence Protocol | sklearn returns tuples constantly |
| 3 | Dictionaries & Sets | Config dicts, feature lookup, deduplication |
| 4 | Comprehensions & Generators | Writing fast, readable data transforms |
| 5 | Functions — First-Class & Higher-Order | `map`, `filter`, custom loss functions |
| 6 | Error Handling & Defensive Code | Production ML pipelines must not silently fail |
| 7 | File I/O & Working with Data Files | Loading CSVs, JSON configs, model artifacts |
| 8 | Python Gotchas — Things That Will Burn We | Mutable defaults, reference semantics, scope |


In [1]:
# This notebook uses only Python standard library — no ML libraries yet
# That's intentional: understand the language before the ecosystem

import os
import json
import csv
import math
import time
import copy
import random
from pathlib import Path
from collections import defaultdict, Counter
from functools import  partial
import sys

print("Python version check:")
print(f"  Python {sys.version}")
print()
print("If We're on Python < 3.8, some f-string and walrus operator features won't work.")
print("Recommendation: use Python 3.10+ for all ML work.")


Python version check:
  Python 3.12.12 (main, Jan 14 2026, 23:36:32) [Clang 21.1.4 ]

If We're on Python < 3.8, some f-string and walrus operator features won't work.
Recommendation: use Python 3.10+ for all ML work.


---
## Section 1 — Types, Variables & the Python Data Model

### Python Is Dynamically Typed — But That Doesn't Mean Untyped

```python
x = 5          # int
x = "hello"    # now it's a str — Python allows this
x = [1, 2, 3]  # now it's a list
```

In Java/C#: the compiler catches type errors at compile time.
In Python: type errors surface at runtime — often deep in our ML pipeline.

**Use type hints** (Python 3.5+) to document intent and catch bugs early:
```python
def train(X: np.ndarray, y: np.ndarray, lr: float = 0.01) -> dict:
    ...
```

### None vs NaN — The Two "Missing" Values

This distinction breaks ML pipelines constantly.

| Value | Type | Meaning | Danger |
|---|---|---|---|
| `None` | `NoneType` | Python's null — "no value" | `None + 1` raises TypeError |
| `np.nan` | `float` | IEEE 754 "not a number" | Silently propagates: `nan + 1 = nan` |
| `pd.NA` | `pd.NA` | Pandas nullable NA | Works in nullable integer arrays |

**Rule:** Use `None` for "this value doesn't exist." `NaN` appears in numerical data when values are missing.


In [2]:
# ── Type system and identity ──────────────────────────────────────────────────
print("=== Python Type System ===")
print()

# Everything is an object — including types themselves
x = 42
print(f"  Value: {x}")
print(f"  Type:  {type(x)}")
print(f"  ID:    {id(x)}  ← memory address of the object")
print()

# type() vs isinstance() — prefer isinstance in production code
# isinstance handles inheritance correctly
print("  isinstance vs type():")
print(f"    isinstance(True, int)  = {isinstance(True, int)}  ← True is a subclass of int!")
print(f"    type(True) == int      = {type(True) == int}       ← misses the subclass")
print(f"    isinstance(True, bool) = {isinstance(True, bool)}")
print()

# ── None semantics ────────────────────────────────────────────────────────────
print("=== None ===")
a = None
print(f"  a is None:     {a is None}")      # use 'is', not '=='
print(f"  a == None:     {a == None}")       # works, but bad style
print(f"  bool(None):    {bool(None)}")      # None is falsy
print(f"  type(None):    {type(None)}")
print()

# The difference between 'is' and '=='
# 'is' checks identity (same object in memory)
# '==' checks equality (same value)
x = [1, 2, 3]
y = [1, 2, 3]
z = x
print("  x = [1,2,3]  y = [1,2,3]  z = x")
print(f"  x == y: {x == y}   ← same value")
print(f"  x is y: {x is y}  ← different objects")
print(f"  x is z: {x is z}   ← z points to same object as x")
print()

# ── NaN behavior — the silent killer ─────────────────────────────────────────
print("=== NaN (float('nan')) ===")
nan = float('nan')
print(f"  nan:           {nan}")
print(f"  nan + 1:       {nan + 1}    ← silently propagates!")
print(f"  nan == nan:    {nan == nan}  ← NaN is not equal to itself!")
print(f"  nan is nan:    {nan is nan}   ← but it's the same object")
print(f"  math.isnan(nan): {math.isnan(nan)}")
print()
print("  This means We can't check for NaN with ==")
print("  Always use math.isnan() or np.isnan() or pd.isna()")
print()

# ── Type coercion surprises ───────────────────────────────────────────────────
print("=== Type Coercion Surprises ===")
print(f"  int('42'):          {int('42')}")
print(f"  float('3.14'):      {float('3.14')}")
print(f"  bool(0):            {bool(0)}   ← 0 is falsy")
print(f"  bool(''):           {bool('')}   ← empty string is falsy")
print(f"  bool([]):           {bool([])}   ← empty list is falsy")
print(f"  bool(0.0):          {bool(0.0)}   ← 0.0 is falsy")
print(f"  bool('False'):      {bool('False')}   ← non-empty string is TRUTHY (gotcha!)")
print()
print("  Gotcha: if x: treats 0, '', [], None, 0.0 as False")
print("  Be explicit: if x is not None:  vs  if x:")


=== Python Type System ===

  Value: 42
  Type:  <class 'int'>
  ID:    4348110624  ← memory address of the object

  isinstance vs type():
    isinstance(True, int)  = True  ← True is a subclass of int!
    type(True) == int      = False       ← misses the subclass
    isinstance(True, bool) = True

=== None ===
  a is None:     True
  a == None:     True
  bool(None):    False
  type(None):    <class 'NoneType'>

  x = [1,2,3]  y = [1,2,3]  z = x
  x == y: True   ← same value
  x is y: False  ← different objects
  x is z: True   ← z points to same object as x

=== NaN (float('nan')) ===
  nan:           nan
  nan + 1:       nan    ← silently propagates!
  nan == nan:    False  ← NaN is not equal to itself!
  nan is nan:    True   ← but it's the same object
  math.isnan(nan): True

  This means We can't check for NaN with ==
  Always use math.isnan() or np.isnan() or pd.isna()

=== Type Coercion Surprises ===
  int('42'):          42
  float('3.14'):      3.14
  bool(0):            Fa

---
## Section 2 — Lists, Tuples & the Sequence Protocol

### Lists vs Tuples — The Real Difference

```python
my_list  = [1, 2, 3]   # mutable — can change
my_tuple = (1, 2, 3)   # immutable — cannot change
```

Beyond mutability, there's a **semantic convention**:
- **List** → homogeneous items of the same type: `[sample1, sample2, sample3]`
- **Tuple** → heterogeneous items forming a record: `(feature_name, value, dtype)`

sklearn follows this convention everywhere:
```python
train_test_split(X, y) → (X_train, X_test, y_train, y_test)  # returns a tuple-like
model.get_params()     → {'alpha': 1.0, 'fit_intercept': True}
cross_val_score(...)   → np.array([0.92, 0.88, 0.91, 0.89, 0.90])
```

### Slicing — Python's Superpower

```python
a = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
a[2:5]    → [2, 3, 4]       # index 2 up to (not including) 5
a[-3:]    → [7, 8, 9]       # last 3 elements
a[::2]    → [0, 2, 4, 6, 8] # every 2nd element
a[::-1]   → [9, 8, 7, ...] # reversed
```

NumPy and Pandas extend this slicing to 2D arrays — We'll use it constantly.


In [3]:
# ── Lists ─────────────────────────────────────────────────────────────────────
print("=== Lists ===")
# Creating
nums    = [1, 2, 3, 4, 5]
mixed   = [1, "hello", 3.14, True, None]   # Python allows mixed types (NumPy doesn't)
nested  = [[1, 2], [3, 4], [5, 6]]         # 2D list (use NumPy for real matrix work)

print(f"  nums:   {nums}")
print(f"  mixed:  {mixed}")
print()

# Slicing — master this, it's everywhere in ML code
a = list(range(10))   # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
print(f"  a          = {a}")
print(f"  a[2:5]     = {a[2:5]}       ← index 2 up to (not including) 5")
print(f"  a[-3:]     = {a[-3:]}         ← last 3 elements")
print(f"  a[::-1]    = {a[::-1]}  ← reversed")
print(f"  a[::2]     = {a[::2]}   ← every 2nd")
print(f"  a[1:8:2]   = {a[1:8:2]}     ← start=1, stop=8, step=2")
print()

# Common list operations in ML pipelines
features = ['age', 'income', 'education', 'experience']
print(f"  Original features: {features}")
print(f"  First 2:           {features[:2]}")
print(f"  All but first:     {features[1:]}")
print(f"  Copy (safe):       {features[:]}  ← features[:] creates a shallow copy")
print()

# Modifying lists
scores = [0.91, 0.88, 0.93, 0.87, 0.90]
scores.append(0.92)       # add one item to end
scores.extend([0.89, 0.94])  # add multiple items
print(f"  Scores after append+extend: {scores}")
print(f"  Max score: {max(scores):.3f}")
print(f"  Sorted:    {sorted(scores, reverse=True)}")   # sorted() returns new list
print()

# ── Tuples ─────────────────────────────────────────────────────────────────────
print("=== Tuples ===")
# Tuple unpacking — used constantly in ML code
X_train, X_test, y_train, y_test = ([1,2], [3,4], [0,1], [1,0])
print(f"  Unpacked: X_train={X_train}, X_test={X_test}")

# Star unpacking
first, *rest = [1, 2, 3, 4, 5]
print(f"  first={first}, rest={rest}")

*head, last = [1, 2, 3, 4, 5]
print(f"  head={head}, last={last}")

# Tuple as dict key (lists can't be dict keys — they're mutable)
cache = {}
cache[(2, 3)] = "result for input (2,3)"   # tuple as key
print(f"  cache: {cache}")

# Swapping without temp variable (Pythonic)
a, b = 10, 20
a, b = b, a
print(f"  After swap: a={a}, b={b}")
print()

# ── enumerate and zip — used in every ML training loop ────────────────────────
print("=== enumerate and zip ===")
features   = ['size', 'bedrooms', 'age']
importance = [0.62, 0.24, 0.14]

# enumerate — when We need both index and value
print("  Feature importance ranking:")
for i, (feat, imp) in enumerate(zip(features, importance), 1):
    print(f"    {i}. {feat:12} {imp:.2%}")

# zip — pairs up iterables; stops at shortest
epochs = [1, 2, 3, 4, 5]
losses = [0.85, 0.62, 0.48, 0.40, 0.37]
accs   = [0.71, 0.81, 0.87, 0.90, 0.91]
print()
print("  Training history:")
print(f"  {'Epoch':>7}  {'Loss':>8}  {'Accuracy':>10}")
for epoch, loss, acc in zip(epochs, losses, accs):
    print(f"  {epoch:>7}  {loss:>8.4f}  {acc:>10.4f}")


=== Lists ===
  nums:   [1, 2, 3, 4, 5]
  mixed:  [1, 'hello', 3.14, True, None]

  a          = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  a[2:5]     = [2, 3, 4]       ← index 2 up to (not including) 5
  a[-3:]     = [7, 8, 9]         ← last 3 elements
  a[::-1]    = [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]  ← reversed
  a[::2]     = [0, 2, 4, 6, 8]   ← every 2nd
  a[1:8:2]   = [1, 3, 5, 7]     ← start=1, stop=8, step=2

  Original features: ['age', 'income', 'education', 'experience']
  First 2:           ['age', 'income']
  All but first:     ['income', 'education', 'experience']
  Copy (safe):       ['age', 'income', 'education', 'experience']  ← features[:] creates a shallow copy

  Scores after append+extend: [0.91, 0.88, 0.93, 0.87, 0.9, 0.92, 0.89, 0.94]
  Max score: 0.940
  Sorted:    [0.94, 0.93, 0.92, 0.91, 0.9, 0.89, 0.88, 0.87]

=== Tuples ===
  Unpacked: X_train=[1, 2], X_test=[3, 4]
  first=1, rest=[2, 3, 4, 5]
  head=[1, 2, 3, 4], last=5
  cache: {(2, 3): 'result for input (2,3)'}
  After 

---
## Section 3 — Dictionaries & Sets

### Dictionaries — The ML Engineer's Best Friend

Dictionaries are everywhere in ML:
- Hyperparameter configs: `{'lr': 0.01, 'epochs': 100, 'batch_size': 32}`
- Model results: `{'train_loss': 0.12, 'val_loss': 0.15, 'r2': 0.91}`
- Feature mappings: `{'neighborhood': 3, 'condition': 2, ...}`
- sklearn `get_params()`, `fit_params`, GridSearchCV parameter grids

### Dict Comprehensions

```python
# Create a mapping from feature name to its index
features = ['age', 'income', 'education']
feat_to_idx = {feat: i for i, feat in enumerate(features)}
# → {'age': 0, 'income': 1, 'education': 2}
```

### Sets — Fast Membership Checks

```python
seen_ids = set()
if sample_id not in seen_ids:
    seen_ids.add(sample_id)
```

`in` on a list is O(n). `in` on a set is O(1). For large datasets, this matters.


In [4]:
# ── Dictionary fundamentals ───────────────────────────────────────────────────
print("=== Dictionaries ===")

# ML use case: hyperparameter config
config = {
    'learning_rate': 0.001,
    'epochs': 100,
    'batch_size': 32,
    'optimizer': 'adam',
    'dropout': 0.3,
    'hidden_layers': [128, 64, 32],
}

# Accessing values
print(f"  Learning rate:  {config['learning_rate']}")
print(f"  Hidden layers:  {config['hidden_layers']}")

# .get() with default — NEVER use d[key] when key might be missing
dropout = config.get('dropout', 0.0)       # returns 0.0 if 'dropout' not in config
momentum = config.get('momentum', 0.9)     # not in config → returns default 0.9
print(f"  Dropout (get):  {dropout}")
print(f"  Momentum (get): {momentum}  ← not in config, used default")
print()

# Iterating — three ways
print("  Dict iteration:")
for key in config:                          # iterate over keys
    if isinstance(config[key], (int, float)):
        print(f"    key: {key:15} = {config[key]}")

print()
for key, value in config.items():           # iterate over key-value pairs
    pass   # this is the idiomatic way

# Dict comprehension — build lookup tables
features = ['GrLivArea', 'OverallQual', 'GarageCars', 'TotalBsmtSF', 'FullBath']
feat_idx  = {feat: i for i, feat in enumerate(features)}
print(f"  Feature index map: {feat_idx}")
print()

# Filtering a dict
important = {k: v for k, v in feat_idx.items() if 'Area' in k or 'SF' in k}
print(f"  Area/SF features: {important}")
print()

# Merging dicts (Python 3.9+ uses |, older uses {**a, **b})
defaults  = {'lr': 0.01, 'epochs': 10, 'batch_size': 32}
overrides = {'epochs': 50, 'batch_size': 64}
merged = {**defaults, **overrides}   # overrides win on conflicts
print(f"  Merged config: {merged}")
print()

# defaultdict — avoid KeyError for counts/accumulation
print("=== defaultdict ===")
model_results = ['ridge', 'lasso', 'ridge', 'elasticnet', 'ridge', 'lasso']
counts = defaultdict(int)   # default value for missing key is int() = 0
for model in model_results:
    counts[model] += 1      # no KeyError if key doesn't exist yet
print(f"  Model counts: {dict(counts)}")

# defaultdict(list) — group items
fold_scores = [(0, 0.91), (1, 0.88), (0, 0.93), (2, 0.87), (1, 0.90), (2, 0.89)]
by_fold = defaultdict(list)
for fold, score in fold_scores:
    by_fold[fold].append(score)
print(f"  Scores by fold: {dict(by_fold)}")
print("  Mean per fold:  ", {f: sum(v)/len(v) for f, v in by_fold.items()})
print()

# ── Counter ───────────────────────────────────────────────────────────────────
print("=== Counter ===")
labels = [0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0]
c = Counter(labels)
total = len(labels)
print(f"  Label distribution: {dict(c)}")
print(f"  Class 0: {c[0]/total:.1%}  Class 1: {c[1]/total:.1%}")
print(f"  Most common: {c.most_common(1)}")
print()

# ── Sets ──────────────────────────────────────────────────────────────────────
print("=== Sets ===")
train_ids = {101, 102, 103, 104, 105, 106}
test_ids  = {104, 105, 106, 107, 108, 109}

print(f"  Train IDs: {sorted(train_ids)}")
print(f"  Test IDs:  {sorted(test_ids)}")
print(f"  Overlap (intersection): {train_ids & test_ids}  ← data leakage check!")
print(f"  Train only:             {train_ids - test_ids}")
print(f"  All IDs (union):        {sorted(train_ids | test_ids)}")
print()
print("  Membership check — O(1) for sets vs O(n) for lists:")
big_list = list(range(100000))
big_set  = set(range(100000))

t0 = time.perf_counter()
_ = 99999 in big_list
t1 = time.perf_counter()
t2 = time.perf_counter()
_ = 99999 in big_set
t3 = time.perf_counter()
print(f"    List membership: {(t1-t0)*1e6:.1f} μs")
print(f"    Set membership:  {(t3-t2)*1e6:.1f} μs  ← much faster")


=== Dictionaries ===
  Learning rate:  0.001
  Hidden layers:  [128, 64, 32]
  Dropout (get):  0.3
  Momentum (get): 0.9  ← not in config, used default

  Dict iteration:
    key: learning_rate   = 0.001
    key: epochs          = 100
    key: batch_size      = 32
    key: dropout         = 0.3

  Feature index map: {'GrLivArea': 0, 'OverallQual': 1, 'GarageCars': 2, 'TotalBsmtSF': 3, 'FullBath': 4}

  Area/SF features: {'GrLivArea': 0, 'TotalBsmtSF': 3}

  Merged config: {'lr': 0.01, 'epochs': 50, 'batch_size': 64}

=== defaultdict ===
  Model counts: {'ridge': 3, 'lasso': 2, 'elasticnet': 1}
  Scores by fold: {0: [0.91, 0.93], 1: [0.88, 0.9], 2: [0.87, 0.89]}
  Mean per fold:   {0: 0.92, 1: 0.89, 2: 0.88}

=== Counter ===
  Label distribution: {0: 8, 1: 6}
  Class 0: 57.1%  Class 1: 42.9%
  Most common: [(0, 8)]

=== Sets ===
  Train IDs: [101, 102, 103, 104, 105, 106]
  Test IDs:  [104, 105, 106, 107, 108, 109]
  Overlap (intersection): {104, 105, 106}  ← data leakage check!
  Train

---
## Section 4 — Comprehensions & Generators

### The Pythonic Way to Transform Data

In Java/C# We'd write a for loop, create a result list, append to it.  
In Python, We use a **comprehension** — one line, no boilerplate.

```python
# Java-style (works but not idiomatic)
squares = []
for x in range(10):
    squares.append(x ** 2)

# Pythonic (preferred)
squares = [x**2 for x in range(10)]
```

### When to Use What

| Pattern | Use for |
|---|---|
| `[... for x in ...]` | List comprehension — build a list |
| `{k: v for ...}` | Dict comprehension — build a lookup |
| `{... for x in ...}` | Set comprehension — unique values |
| `(... for x in ...)` | Generator expression — lazy, memory-efficient |

### Generators — For Large Data

A **generator** computes values on demand instead of storing all of them in memory.  
Critical when iterating over datasets that don't fit in RAM.

```python
# List comprehension: builds the whole list in memory
all_squares = [x**2 for x in range(1_000_000)]   # uses ~8MB

# Generator: computes one at a time, uses almost no memory
gen_squares = (x**2 for x in range(1_000_000))    # uses ~100 bytes
```


In [5]:
import sys

# ── List comprehensions ───────────────────────────────────────────────────────
print("=== List Comprehensions ===")

# Basic
squares = [x**2 for x in range(10)]
print(f"  Squares:     {squares}")

# With condition (filter)
even_squares = [x**2 for x in range(10) if x % 2 == 0]
print(f"  Even squares: {even_squares}")

# Nested (flattening a 2D list)
matrix  = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat    = [val for row in matrix for val in row]
print(f"  Flattened:   {flat}")
print()

# ML use case: normalize a list of scores
raw_scores = [0.3, 0.8, 0.5, 1.2, 0.9, 0.4]
min_s, max_s = min(raw_scores), max(raw_scores)
normalized = [(s - min_s) / (max_s - min_s) for s in raw_scores]
print(f"  Raw scores:   {raw_scores}")
print(f"  Normalized:   {[round(s, 3) for s in normalized]}")
print()

# ML use case: extract specific keys from a list of dicts
results = [
    {'model': 'ridge',      'r2': 0.92, 'rmse': 0.18},
    {'model': 'lasso',      'r2': 0.89, 'rmse': 0.22},
    {'model': 'elasticnet', 'r2': 0.91, 'rmse': 0.19},
]
r2_scores = [r['r2'] for r in results]
best      = max(results, key=lambda r: r['r2'])
print(f"  R² scores: {r2_scores}")
print(f"  Best model: {best['model']} (R²={best['r2']})")
print()

# ── Dict comprehensions ───────────────────────────────────────────────────────
print("=== Dict Comprehensions ===")

# Feature importance lookup
features  = ['GrLivArea', 'OverallQual', 'GarageCars', 'YearBuilt', 'TotalBsmtSF']
importances = [0.34, 0.28, 0.15, 0.13, 0.10]
feat_imp  = {f: imp for f, imp in zip(features, importances)}
print(f"  Feature importances: {feat_imp}")

# Filter to top features
top_feats = {f: imp for f, imp in feat_imp.items() if imp > 0.15}
print(f"  Top features:        {top_feats}")

# Invert a mapping
idx_to_feat = {i: f for f, i in enumerate(features)}
print(f"  Index → feature:     {idx_to_feat}")
print()

# ── Generators ────────────────────────────────────────────────────────────────
print("=== Generators ===")

def batch_generator(data, batch_size):
    # Yields data in chunks — fundamental pattern for ML training
    # 'yield' makes this a generator function
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]

dataset = list(range(20))   # pretend this is 20 training samples
print(f"  Dataset: {dataset}")
print("  Batches of 6:")
for i, batch in enumerate(batch_generator(dataset, 6)):
    print(f"    Batch {i}: {batch}")
print()

# Memory comparison
list_comp = [x**2 for x in range(100000)]
gen_expr  = (x**2 for x in range(100000))
print(f"  List of 100k squares:      {sys.getsizeof(list_comp):>8} bytes")
print(f"  Generator for 100k squares: {sys.getsizeof(gen_expr):>8} bytes  ← massive difference")
print()

# ── Walrus operator := (Python 3.8+) ─────────────────────────────────────────
print("=== Walrus Operator := (assign-and-test) ===")
# Useful in while loops for ML training
data_stream = iter([0.9, 0.7, 0.5, 0.3, 0.18, 0.12, 0.09])
threshold   = 0.15
print("  Training until loss < 0.15:")
step = 0
while (loss := next(data_stream, None)) is not None:
    step += 1
    print(f"    Step {step}: loss={loss:.2f}  {'STOP' if loss < threshold else ''}")
    if loss < threshold:
        break


=== List Comprehensions ===
  Squares:     [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
  Even squares: [0, 4, 16, 36, 64]
  Flattened:   [1, 2, 3, 4, 5, 6, 7, 8, 9]

  Raw scores:   [0.3, 0.8, 0.5, 1.2, 0.9, 0.4]
  Normalized:   [0.0, 0.556, 0.222, 1.0, 0.667, 0.111]

  R² scores: [0.92, 0.89, 0.91]
  Best model: ridge (R²=0.92)

=== Dict Comprehensions ===
  Feature importances: {'GrLivArea': 0.34, 'OverallQual': 0.28, 'GarageCars': 0.15, 'YearBuilt': 0.13, 'TotalBsmtSF': 0.1}
  Top features:        {'GrLivArea': 0.34, 'OverallQual': 0.28}
  Index → feature:     {'GrLivArea': 0, 'OverallQual': 1, 'GarageCars': 2, 'YearBuilt': 3, 'TotalBsmtSF': 4}

=== Generators ===
  Dataset: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  Batches of 6:
    Batch 0: [0, 1, 2, 3, 4, 5]
    Batch 1: [6, 7, 8, 9, 10, 11]
    Batch 2: [12, 13, 14, 15, 16, 17]
    Batch 3: [18, 19]

  List of 100k squares:        800984 bytes
  Generator for 100k squares:      200 bytes  ← massive differ

---
## Section 5 — Functions as First-Class Objects

Python functions are **objects** — We can pass them as arguments,  
return them from other functions, and store them in data structures.

This is fundamental to ML frameworks. Every callback, custom loss,  
metric function, and transform pipeline relies on this.

```python
# Functions can be passed as arguments
def apply(func, data):
    return func(data)

apply(math.sqrt, 16)   # → 4.0
apply(np.log1p, 16)    # → 2.833...
```

### Lambda — Anonymous Functions

```python
# Full function
def square(x): return x**2

# Lambda — same thing, one line
square = lambda x: x**2

# Common use: sorting by a key
sorted(results, key=lambda r: r['val_loss'])
max(models, key=lambda m: m.score(X_test, y_test))
```

### map, filter, reduce

```python
map(func, iterable)     → lazy, apply func to each element
filter(func, iterable)  → keep elements where func returns True
reduce(func, iterable)  → fold: combine all elements into one value
```


In [6]:
# ── First-class functions ─────────────────────────────────────────────────────
print("=== Functions as Objects ===")

def relu(x):
    return max(0, x)

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def tanh(x):
    return math.tanh(x)

# Store functions in a dict — common pattern for selecting activations
activations = {
    'relu':    relu,
    'sigmoid': sigmoid,
    'tanh':    tanh,
}

config = {'activation': 'relu'}
activate = activations[config['activation']]   # look up the function
print(f"  Activation function: {config['activation']}")
print(f"  relu(-1) = {activate(-1)}")
print(f"  relu(0)  = {activate(0)}")
print(f"  relu(2)  = {activate(2)}")
print()

# Functions as arguments — custom sorting
model_runs = [
    {'name': 'run_01', 'val_r2': 0.88, 'train_r2': 0.95, 'time_s': 12},
    {'name': 'run_02', 'val_r2': 0.92, 'train_r2': 0.94, 'time_s': 8},
    {'name': 'run_03', 'val_r2': 0.91, 'train_r2': 0.97, 'time_s': 25},
    {'name': 'run_04', 'val_r2': 0.85, 'train_r2': 0.87, 'time_s': 5},
]
# Sort by val_r2 descending
by_val_r2    = sorted(model_runs, key=lambda r: r['val_r2'],  reverse=True)
# Sort by training time ascending
by_time      = sorted(model_runs, key=lambda r: r['time_s'])
# Sort by gap (overfitting) ascending
by_gap       = sorted(model_runs, key=lambda r: r['train_r2'] - r['val_r2'])

print("  By val R² (best first):")
for r in by_val_r2:
    print(f"    {r['name']}  val_r2={r['val_r2']}  gap={r['train_r2']-r['val_r2']:.3f}")
print()

# ── Lambda ────────────────────────────────────────────────────────────────────
print("=== Lambda Functions ===")

# Common ML lambdas
normalize = lambda x, mean, std: (x - mean) / std
clip      = lambda x, lo, hi: max(lo, min(hi, x))
rmsle     = lambda y_true, y_pred: math.sqrt(
                sum((math.log1p(p) - math.log1p(a))**2
                    for a, p in zip(y_true, y_pred)) / len(y_true))

actuals   = [200000, 150000, 300000, 120000]
preds     = [210000, 145000, 315000, 118000]
print(f"  RMSLE: {rmsle(actuals, preds):.5f}")
print()

# ── map and filter ────────────────────────────────────────────────────────────
print("=== map and filter ===")
raw_values = [-2.5, 1.3, -0.8, 3.1, -1.5, 2.2, 0.4]

# Apply ReLU to all values
relu_values = list(map(relu, raw_values))
print(f"  Input:  {raw_values}")
print(f"  ReLU:   {relu_values}")

# Filter to positive only
positives = list(filter(lambda x: x > 0, raw_values))
print(f"  Positive: {positives}")

# Note: in modern Python, list comprehensions are usually preferred over map/filter
# relu_values = [relu(x) for x in raw_values]  ← more readable
print()

# ── partial — fix some arguments ─────────────────────────────────────────────
print("=== partial (partial application) ===")

# We have a function: normalize(x, mean, std)
# We want to create a version with mean=0, std=1 already fixed
def normalize_1(x, mean, std):
    return (x - mean) / std

# In sklearn, We often need to pass a function with no arguments
# partial lets We pre-fill some arguments
normalize_standard = partial(normalize, mean=0.0, std=1.0)
normalize_custom   = partial(normalize_1, mean=100.0, std=15.0)

data = [85, 100, 115, 70, 130]
print(f"  Data: {data}")
print(f"  Standard normalized: {[round(normalize_standard(x), 2) for x in data]}")
print(f"  Custom normalized:   {[round(normalize_custom(x), 2) for x in data]}")


=== Functions as Objects ===
  Activation function: relu
  relu(-1) = 0
  relu(0)  = 0
  relu(2)  = 2

  By val R² (best first):
    run_02  val_r2=0.92  gap=0.020
    run_03  val_r2=0.91  gap=0.060
    run_01  val_r2=0.88  gap=0.070
    run_04  val_r2=0.85  gap=0.020

=== Lambda Functions ===
  RMSLE: 0.03935

=== map and filter ===
  Input:  [-2.5, 1.3, -0.8, 3.1, -1.5, 2.2, 0.4]
  ReLU:   [0, 1.3, 0, 3.1, 0, 2.2, 0.4]
  Positive: [1.3, 3.1, 2.2, 0.4]

=== partial (partial application) ===
  Data: [85, 100, 115, 70, 130]
  Standard normalized: [85.0, 100.0, 115.0, 70.0, 130.0]
  Custom normalized:   [-1.0, 0.0, 1.0, -2.0, 2.0]


---
## Section 6 — Error Handling & Defensive Code

### Why This Matters in ML Pipelines

A training pipeline might run for 8 hours.  
If it crashes at hour 7 due to an unhandled exception, We've wasted that time.

Good ML code:
- Validates inputs before starting long computations
- Handles expected failure modes gracefully
- Logs what went wrong with enough context to debug
- Uses `finally` to clean up resources even if something fails

### Python Exception Hierarchy

```
BaseException
├── SystemExit
├── KeyboardInterrupt
└── Exception
    ├── ValueError    ← wrong value type (e.g. negative array size)
    ├── TypeError     ← wrong type (e.g. string where int expected)
    ├── KeyError      ← missing dict key
    ├── IndexError    ← list index out of range
    ├── FileNotFoundError (OSError) ← file missing
    ├── ZeroDivisionError
    └── ...
```


In [7]:
import tempfile

# ── try / except / else / finally ────────────────────────────────────────────
print("=== Exception Handling ===")

def load_dataset(filepath: str) -> list:
    # Real-world pattern: function that can fail in multiple ways
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        print(f"  ERROR: File not found: {filepath}")
        return []
    except json.JSONDecodeError as e:
        print(f"  ERROR: Invalid JSON in {filepath}: {e}")
        return []
    except Exception as e:
        # Catch-all for unexpected errors — log and re-raise
        print(f"  UNEXPECTED ERROR: {type(e).__name__}: {e}")
        raise   # don't silently swallow unexpected errors

# Test it
print("  Trying to load nonexistent file:")
data = load_dataset('/nonexistent/path/data.json')
print(f"  Got: {data}")
print()

# ── else and finally ──────────────────────────────────────────────────────────
print("=== else and finally ===")
# else: runs only if NO exception occurred
# finally: runs ALWAYS (cleanup)

def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print(f"  Cannot divide {a} by zero")
        return None
    else:
        # Only runs if no exception
        print(f"  {a} / {b} = {result:.4f}")
        return result
    finally:
        # ALWAYS runs — use for cleanup (close files, release locks, etc.)
        pass  # nothing to clean up here

safe_divide(10, 3)
safe_divide(10, 0)
print()

# ── Input validation — assert and custom exceptions ────────────────────────
print("=== Input Validation ===")

class InvalidDataError(Exception):
    # Custom exception — makes our code self-documenting
    pass

def train_model(X: list, y: list, learning_rate: float):
    # Validate inputs before doing anything expensive
    if len(X) == 0:
        raise ValueError("Training data X cannot be empty")
    if len(X) != len(y):
        raise ValueError(f"X and y must have same length: {len(X)} != {len(y)}")
    if not (0 < learning_rate < 1):
        raise ValueError(f"learning_rate must be between 0 and 1, got {learning_rate}")
    if any(v is None for v in y):
        raise InvalidDataError("y contains None values — check our data pipeline")

    print(f"  Training on {len(X)} samples with lr={learning_rate}")
    return {'status': 'trained'}

# Valid call
train_model([1, 2, 3], [0, 1, 1], learning_rate=0.01)

# Invalid calls — caught before any work is done
for X_arg, y_arg, kwargs in [
    ([], [], {}),
    ([1, 2, 3], [0, 1], {}),
    ([1, 2, 3], [0, 1, 1], {'learning_rate': 1.5}),
]:
    try:
        train_model(
            X_arg,
            y_arg,
            kwargs.get('learning_rate', 0.01)
        )
    except (ValueError, InvalidDataError) as e:
        print(f"  Caught: {type(e).__name__}: {e}")

print()

# ── Context managers — the with statement ─────────────────────────────────────
print("=== Context Managers (with statement) ===")
# Context managers ensure resources are released even if an error occurs

# File handling — ALWAYS use with for files

# Write a temp CSV
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    tmpfile = f.name
    f.write("feature1,feature2,label\n")
    f.write("1.0,2.0,1\n")
    f.write("3.0,4.0,0\n")

# Read it back
with open(tmpfile, 'r') as f:
    for line in f:
        print(f"  Read: {line.rstrip()}")

os.unlink(tmpfile)  # clean up
print("  File automatically closed by context manager ✓")


=== Exception Handling ===
  Trying to load nonexistent file:
  ERROR: File not found: /nonexistent/path/data.json
  Got: []

=== else and finally ===
  10 / 3 = 3.3333
  Cannot divide 10 by zero

=== Input Validation ===
  Training on 3 samples with lr=0.01
  Caught: ValueError: Training data X cannot be empty
  Caught: ValueError: X and y must have same length: 3 != 2
  Caught: ValueError: learning_rate must be between 0 and 1, got 1.5

=== Context Managers (with statement) ===
  Read: feature1,feature2,label
  Read: 1.0,2.0,1
  Read: 3.0,4.0,0
  File automatically closed by context manager ✓


---
## Section 7 — File I/O & Working with Data Files

### The Files We'll Work With in ML

| Format | Use case | Python module |
|---|---|---|
| `.csv` | Tabular datasets | `csv`, `pandas` |
| `.json` | Configs, API responses, model metadata | `json` |
| `.txt` | Text corpora, logs | built-in `open()` |
| `.pkl` / `.joblib` | Saved model objects | `pickle`, `joblib` |
| `.npy` / `.npz` | Saved NumPy arrays | `numpy` |
| `.parquet` | Large tabular datasets (columnar) | `pandas` |

### The `pathlib.Path` — Always Use This Instead of `os.path`

```python
# Old way (fragile, OS-specific)
path = '/datasets/' + filename + '.csv'

# New way (pathlib — OS-agnostic, readable)
from pathlib import Path
data_dir = Path('/datasets')
path     = data_dir / filename                # path / path works!
csv_path = data_dir / f'{filename}.csv'
```


In [8]:
import pickle

# ── pathlib — the modern way ──────────────────────────────────────────────────
print("=== pathlib.Path ===")

base_dir   = Path('/datasets')
model_dir  = base_dir / 'models'
data_file  = base_dir / 'house_prices_train.csv'
config_file = base_dir / 'config.json'

print(f"  Base dir:    {base_dir}")
print(f"  Data file:   {data_file}")
print(f"  Stem:        {data_file.stem}")       # filename without extension
print(f"  Suffix:      {data_file.suffix}")     # '.csv'
print(f"  Parent:      {data_file.parent}")     # '/datasets'
print(f"  Exists:      {data_file.exists()}")   # True if file exists
print()

# ── JSON — configs, hyperparameters, experiment tracking ──────────────────────
print("=== JSON ===")

experiment = {
    'experiment_id': 'exp_001',
    'model': 'Ridge',
    'hyperparams': {'alpha': 1.0, 'fit_intercept': True},
    'metrics': {'train_r2': 0.91, 'val_r2': 0.88, 'val_rmsle': 0.14},
    'features': ['GrLivArea', 'OverallQual', 'GarageCars'],
    'timestamp': '2026-02-26T10:30:00',
}

# Save to JSON
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json_path = f.name
    json.dump(experiment, f, indent=2)  # indent=2 for human-readable
print(f"  Saved experiment to: {json_path}")

# Load from JSON
with open(json_path, 'r') as f:
    loaded = json.load(f)
print(f"  Loaded model:    {loaded['model']}")
print(f"  Best val R²:     {loaded['metrics']['val_r2']}")
print(f"  Hyperparams:     {loaded['hyperparams']}")
print()

# json.dumps / json.loads — convert to/from string (useful for APIs, logging)
json_str = json.dumps(experiment['metrics'])
print(f"  As string: {json_str}")
parsed   = json.loads(json_str)
print(f"  Parsed:    {parsed}")
os.unlink(json_path)
print()

# ── CSV reading ───────────────────────────────────────────────────────────────
print("=== CSV ===")
# Create a small CSV to work with
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, newline='') as f:
    csv_path = f.name
    writer = csv.DictWriter(f, fieldnames=['id', 'feature1', 'feature2', 'label'])
    writer.writeheader()
    for i in range(5):
        writer.writerow({'id': i, 'feature1': round(random.gauss(0, 1), 3),
                        'feature2': round(random.gauss(5, 2), 3),
                        'label': random.randint(0, 1)})

# Read as list of dicts (DictReader — each row is a dict)
with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    rows   = list(reader)

print(f"  Loaded {len(rows)} rows:")
for row in rows:
    print(f"    {row}")

# Convert to lists for processing
features = [[float(r['feature1']), float(r['feature2'])] for r in rows]
labels   = [int(r['label']) for r in rows]
print(f"  Features: {features}")
print(f"  Labels:   {labels}")
os.unlink(csv_path)
print()

# ── pickle — save and load Python objects (models, preprocessors) ─────────────
print("=== pickle (model persistence) ===")
# In real ML: use joblib.dump(model, path) for sklearn models (better for numpy arrays)
# pickle works for any Python object

model_artifact = {
    'type': 'Ridge',
    'alpha': 1.0,
    'coef': [0.34, 0.28, 0.15, -0.08],  # pretend these are learned weights
    'intercept': 12.3,
    'feature_names': ['GrLivArea', 'OverallQual', 'GarageCars', 'HouseAge'],
    'scaler_mean': [1500, 7.0, 2.0, 20.0],
    'scaler_std':  [400,  1.5, 0.8, 15.0],
}

with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
    pkl_path = f.name
    pickle.dump(model_artifact, f)

with open(pkl_path, 'rb') as f:
    loaded_model = pickle.load(f)

print("  Saved and loaded model artifact:")
print(f"    Type:     {loaded_model['type']}")
print(f"    Alpha:    {loaded_model['alpha']}")
print(f"    Features: {loaded_model['feature_names']}")
os.unlink(pkl_path)


=== pathlib.Path ===
  Base dir:    /datasets
  Data file:   /datasets/house_prices_train.csv
  Stem:        house_prices_train
  Suffix:      .csv
  Parent:      /datasets
  Exists:      False

=== JSON ===
  Saved experiment to: /var/folders/q1/9498s9914m31_cnj__snl5xm0000gn/T/tmp5p3b_1h_.json
  Loaded model:    Ridge
  Best val R²:     0.88
  Hyperparams:     {'alpha': 1.0, 'fit_intercept': True}

  As string: {"train_r2": 0.91, "val_r2": 0.88, "val_rmsle": 0.14}
  Parsed:    {'train_r2': 0.91, 'val_r2': 0.88, 'val_rmsle': 0.14}

=== CSV ===
  Loaded 5 rows:
    {'id': '0', 'feature1': '0.003', 'feature2': '5.681', 'label': '0'}
    {'id': '1', 'feature1': '2.577', 'feature2': '6.153', 'label': '1'}
    {'id': '2', 'feature1': '1.311', 'feature2': '4.475', 'label': '0'}
    {'id': '3', 'feature1': '0.904', 'feature2': '2.514', 'label': '0'}
    {'id': '4', 'feature1': '-0.368', 'feature2': '5.908', 'label': '0'}
  Features: [[0.003, 5.681], [2.577, 6.153], [1.311, 4.475], [0.904, 2.

---
## Section 8 — Python Gotchas That Will Burn We

These are the bugs that waste experienced engineers' hours.  
All are specific to Python's semantics — if We're coming from Java/C#, these will surprise We.

### 1. Mutable Default Arguments
### 2. Reference Semantics — Assignment ≠ Copy
### 3. Late Binding in Closures
### 4. Integer Division
### 5. Floating-Point Precision
### 6. Variable Scope (LEGB rule)


In [9]:
print("=" * 60)
print("  GOTCHA 1: Mutable Default Arguments")
print("=" * 60)
print()

# THE BUG — default argument is created ONCE, not on each call
def add_result_BAD(score, results=[]):   # ← DANGER: list created once
    results.append(score)
    return results

r1 = add_result_BAD(0.91)
r2 = add_result_BAD(0.88)
r3 = add_result_BAD(0.93)
print(f"  BAD: r1={r1}  r2={r2}  r3={r3}")
print("  All three point to the same list! The default accumulates across calls.")
print()

# THE FIX — use None as default, create inside function
def add_result_GOOD(score, results=None):
    if results is None:
        results = []        # new list created on each call
    results.append(score)
    return results

r1 = add_result_GOOD(0.91)
r2 = add_result_GOOD(0.88)
print(f"  GOOD: r1={r1}  r2={r2}  ← separate lists")
print()

print("=" * 60)
print("  GOTCHA 2: Reference Semantics (Assignment ≠ Copy)")
print("=" * 60)
print()

# Lists are passed by reference — modifying a copy modifies the original
original_features = ['age', 'income', 'education']
features_copy     = original_features   # NOT a copy — same object!
features_copy.append('experience')

print(f"  original_features: {original_features}")
print("  Surprise! Original was modified because 'copy' was just another reference.")
print()

# FIX 1: Shallow copy
original   = [1, 2, 3, 4, 5]
shallow    = original[:]          # or list(original) or original.copy()
shallow.append(99)
print(f"  Shallow copy — original: {original}  copy: {shallow}  ✓")

# FIX 2: Deep copy (needed for nested structures)
nested_original = [[1, 2], [3, 4], [5, 6]]
nested_shallow  = nested_original[:]         # shallow copy
nested_deep     = copy.deepcopy(nested_original)

nested_shallow[0].append(99)   # modifies original too!
nested_deep[0].append(88)      # does NOT modify original

print(f"  nested_original after shallow[0].append(99): {nested_original}")
print(f"  nested_original after deep[0].append(88):    {nested_original}  ← unaffected ✓")
print()

print("=" * 60)
print("  GOTCHA 3: Late Binding in Closures")
print("=" * 60)
print()

# THE BUG
funcs_bad = [lambda x: x * i for i in range(5)]
results   = [f(2) for f in funcs_bad]
print(f"  Expected: [0, 2, 4, 6, 8]  Got: {results}")
print("  All use i=4 because lambda captures the variable, not the value")
print()

# THE FIX — capture value with default argument
funcs_good = [lambda x, i=i: x * i for i in range(5)]
results    = [f(2) for f in funcs_good]
print(f"  Fixed:    {results}  ✓")
print()

print("=" * 60)
print("  GOTCHA 4: Integer Division")
print("=" * 60)
print()
a, b = 7, 2
print(f"  Python 3:  7 / 2  = {7/2}   ← always float")
print(f"  Python 3:  7 // 2 = {7//2}  ← floor division (integer)")
print("  Python 2 GOTCHA:  7 / 2 = 3  (integer division) — this code might still exist!")
print("  In ML:  batch_count = len(data) // batch_size  ← use // for index math")
print()

print("=" * 60)
print("  GOTCHA 5: Floating-Point Precision")
print("=" * 60)
print()
print("  0.1 + 0.2 == 0.3 ?  {0.1 + 0.2 == 0.3}")
print("  0.1 + 0.2          = {0.1 + 0.2}")
print("  Use math.isclose() for float comparisons:")
print("  math.isclose(0.1 + 0.2, 0.3)  = {math.isclose(0.1 + 0.2, 0.3)}")
print()
print("  In ML: this is why loss.item() sometimes shows 0.9999999 instead of 1.0")
print("  Always use math.isclose() or np.allclose() for float equality checks")
print()

print("=" * 60)
print("  GOTCHA 6: Python Scope (LEGB)")
print("=" * 60)
print()
# LEGB: Local → Enclosing → Global → Built-in
x = 'global'

def outer():
    x = 'enclosing'
    def inner():
        # x = 'local'       # if this line is here, x is local
        print(f"    inner sees x = '{x}'")  # sees enclosing scope
    inner()

outer()
print()

# The nonlocal keyword — to modify enclosing scope variable
def make_counter():
    count = 0
    def increment():
        nonlocal count      # without this, count += 1 would fail (UnboundLocalError)
        count += 1
        return count
    return increment

counter = make_counter()
print(f"  Counter: {counter()}, {counter()}, {counter()}")
print("  This pattern is used in callbacks, closures, and custom metrics")


  GOTCHA 1: Mutable Default Arguments

  BAD: r1=[0.91, 0.88, 0.93]  r2=[0.91, 0.88, 0.93]  r3=[0.91, 0.88, 0.93]
  All three point to the same list! The default accumulates across calls.

  GOOD: r1=[0.91]  r2=[0.88]  ← separate lists

  GOTCHA 2: Reference Semantics (Assignment ≠ Copy)

  original_features: ['age', 'income', 'education', 'experience']
  Surprise! Original was modified because 'copy' was just another reference.

  Shallow copy — original: [1, 2, 3, 4, 5]  copy: [1, 2, 3, 4, 5, 99]  ✓
  nested_original after shallow[0].append(99): [[1, 2, 99], [3, 4], [5, 6]]
  nested_original after deep[0].append(88):    [[1, 2, 99], [3, 4], [5, 6]]  ← unaffected ✓

  GOTCHA 3: Late Binding in Closures

  Expected: [0, 2, 4, 6, 8]  Got: [8, 8, 8, 8, 8]
  All use i=4 because lambda captures the variable, not the value

  Fixed:    [0, 2, 4, 6, 8]  ✓

  GOTCHA 4: Integer Division

  Python 3:  7 / 2  = 3.5   ← always float
  Python 3:  7 // 2 = 3  ← floor division (integer)
  Python 2 G

---
## Summary — Python Basics for ML Engineers

### The 10 Things That Matter Most

1. **`is` vs `==`** — use `is` for `None`, `==` for everything else
2. **`isinstance()` not `type()`** — handles inheritance correctly
3. **`dict.get(key, default)`** — never use `dict[key]` when key might be missing
4. **Comprehensions** — `[f(x) for x in data if condition]` is idiomatic Python
5. **`with open(...) as f:`** — always use context managers for files
6. **Mutable default args** — use `None`, create inside function
7. **Assignment ≠ copy** — use `[:]` or `copy.deepcopy()` to copy lists
8. **`//` for integer division** — `/` always returns float in Python 3
9. **`math.isclose()`** — never compare floats with `==`
10. **`try/except/finally`** — catch specific exceptions, clean up in `finally`

### Next Notebook
`oop.ipynb` — Classes, inheritance, and how sklearn's API is designed
